# FashionStyle14 LLaVA Caption Generation

## Overview
This notebook uses LLaVA (Large Language and Vision Assistant) to generate detailed text descriptions for fashion images in the FashionStyle14 dataset.

## Why LLaVA?
- **Free and open-source**: No API costs
- **Good quality**: Competitive with commercial models
- **Runs locally**: Complete privacy and control
- **GPU optimized**: Efficient for batch processing

## Workflow:
1. **Setup LLaVA**: Install and configure the model
2. **Load Dataset**: Use the complete dataset from our previous analysis
3. **Generate Captions**: Process images in batches
4. **Save Results**: Store captions for multi-modal training

## Requirements:
- GPU with at least 8GB VRAM (recommended)
- CUDA-compatible GPU
- Sufficient disk space for model weights (~13GB)


## 1. Setup and Installation


In [1]:
import torch
import pandas as pd
import numpy as np
from PIL import Image
import os
import json
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Total VRAM: {total_memory:.1f} GB")
    
    # Display current GPU memory usage
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    print(f"Current GPU memory - Allocated: {allocated:.2f} GB, Reserved: {reserved:.2f} GB")
else:
    print("Warning: No GPU detected. LLaVA will run on CPU (very slow)")

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Function to monitor GPU memory (useful during processing)
def print_gpu_memory():
    """Print current GPU memory usage"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1024**3
        reserved = torch.cuda.memory_reserved(0) / 1024**3
        print(f"GPU Memory - Allocated: {allocated:.2f} GB, Reserved: {reserved:.2f} GB")
    else:
        print("No GPU available")


CUDA available: True
GPU: NVIDIA GeForce RTX 4090
Total VRAM: 23.5 GB
Current GPU memory - Allocated: 0.00 GB, Reserved: 0.00 GB
Using device: cuda


In [2]:
# Diagnostic: Check LLaVA installation and Python path
# Run this cell if you're getting "ModuleNotFoundError: No module named 'llava'"

import sys
import os

print("Current Python executable:", sys.executable)
print("\nPython path:")
for p in sys.path:
    print(f"  {p}")

print("\nChecking for LLaVA installation...")

# Check if LLaVA is in current directory
llava_paths = [
    "./LLaVA",
    "../LLaVA",
    os.path.expanduser("~/LLaVA"),
]

for path in llava_paths:
    abs_path = os.path.abspath(path)
    if os.path.exists(abs_path):
        print(f"✓ Found LLaVA directory at: {abs_path}")
        # Add to path if not already there
        if abs_path not in sys.path:
            sys.path.insert(0, abs_path)
            print(f"  Added to sys.path")
        
        # Check if it's properly installed
        llava_init = os.path.join(abs_path, "llava", "__init__.py")
        if os.path.exists(llava_init):
            print(f"  ✓ Found llava package")
        else:
            print(f"  ✗ llava package not found in {abs_path}")

# Try importing
print("\nAttempting to import LLaVA...")
try:
    import llava
    print(f"✓ LLaVA imported successfully from: {llava.__file__}")
except ImportError as e:
    print(f"✗ Import failed: {e}")
    print("\nTry one of these solutions:")
    print("\nOption 1: Install LLaVA by running the installation cell above")
    print("\nOption 2: If LLaVA is installed elsewhere, add it to path:")
    print("  import sys")
    print("  sys.path.append('/path/to/LLaVA')")
    print("\nOption 3: If LLaVA repo exists in current directory:")
    print("  import sys")
    print("  sys.path.insert(0, './LLaVA')")


Current Python executable: /home/ding-zhang/anaconda3/envs/llava_env/bin/python

Python path:
  /home/ding-zhang/anaconda3/envs/llava_env/lib/python310.zip
  /home/ding-zhang/anaconda3/envs/llava_env/lib/python3.10
  /home/ding-zhang/anaconda3/envs/llava_env/lib/python3.10/lib-dynload
  
  /home/ding-zhang/anaconda3/envs/llava_env/lib/python3.10/site-packages
  __editable__.llava-1.2.2.post1.finder.__path_hook__

Checking for LLaVA installation...
✓ Found LLaVA directory at: /home/ding-zhang/Dongmei/DATA255/Project/LLaVA
  Added to sys.path
  ✓ Found llava package

Attempting to import LLaVA...
✓ LLaVA imported successfully from: /home/ding-zhang/Dongmei/DATA255/Project/LLaVA/llava/__init__.py


## 2. Load LLaVA Model


In [3]:
from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path
from llava.conversation import conv_templates
from llava.utils import disable_torch_init
import warnings

# Disable torch initialization warnings
disable_torch_init()

# Suppress transformers warnings about model type mismatch (common with LLaVA)
warnings.filterwarnings("ignore", message=".*You are using a model of type.*")

# Clear GPU cache before loading model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory before loading: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

# Load LLaVA model (this will download ~13GB of weights on first run)
model_path = "liuhaotian/llava-v1.5-7b"  # 7B parameter model
print(f"Loading LLaVA model: {model_path}")
print("This may take several minutes on first run...")

# Load model and processor
model_name = get_model_name_from_path(model_path)
tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=model_path,
    model_base=None,
    model_name=model_name,
    load_8bit=False,  # Set to True if you have memory issues
    load_4bit=False,  # Set to True for even more memory savings
    device_map="auto"
)

# Ensure model is on correct device (if device_map="auto" doesn't handle it)
if torch.cuda.is_available() and hasattr(model, 'to'):
    # Model should already be on GPU with device_map="auto", but verify
    print(f"GPU memory after loading: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"GPU memory cached: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")

print("Model loaded successfully!")
print(f"Model name: {model_name}")
print(f"Context length: {context_len}")


GPU memory before loading: 0.00 GB
Loading LLaVA model: liuhaotian/llava-v1.5-7b
This may take several minutes on first run...


You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

GPU memory after loading: 13.78 GB
GPU memory cached: 13.79 GB
Model loaded successfully!
Model name: llava-v1.5-7b
Context length: 2048


## 3. Load Dataset


In [4]:
# ============================================================================
# CONFIGURATION: Specify which style folders to process
# ============================================================================
# Option 1: Process specific folders (recommended for batch processing)
# Example: process only 'conservative' folder
style_folders_to_process = ['gal', 'girlish', 'kireime-casual']  # Change this list as needed

# Option 2: Process all folders (set to None)
# style_folders_to_process = None  # Process all style folders

# ============================================================================

# Load the complete dataset from our previous analysis
try:
    # Try to load from the complete dataset CSV we created
    complete_dataset = pd.read_csv('complete_dataset.csv', header=None, names=['image_path'])
    
    # Extract style categories from paths
    complete_dataset['style'] = complete_dataset['image_path'].apply(
        lambda x: x.split('/')[1] if '/' in x else x.split('\\')[1]
    )
    
    # Filter by specified folders if provided
    if style_folders_to_process is not None:
        complete_dataset = complete_dataset[complete_dataset['style'].isin(style_folders_to_process)]
        print(f"Filtered dataset to specified folders: {style_folders_to_process}")
    
    print(f"Loaded complete dataset: {len(complete_dataset)} images")
    print(f"Style categories: {sorted(complete_dataset['style'].unique())}")
    
except FileNotFoundError:
    print("Complete dataset CSV not found. Discovering dataset from folder...")
    
    # Fallback: discover dataset from folder
    def discover_dataset(base_path='dataset', style_folders=None):
        """
        Discover all images in the dataset folder structure
        
        Args:
            base_path: Base directory containing style folders (default: 'dataset')
            style_folders: List of specific style folder names to process, or None for all
        
        Returns:
            DataFrame with columns: image_path, style
        """
        all_images = []
        if not os.path.exists(base_path):
            raise FileNotFoundError(f"Dataset folder '{base_path}' not found!")
        
        # Check if base_path directly contains images (single folder case)
        direct_images = [f for f in os.listdir(base_path) 
                        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        
        if direct_images:
            # Single folder with images - use folder name as style
            style_name = os.path.basename(base_path.rstrip('/'))
            # Check if this style should be included
            if style_folders is None or style_name in style_folders:
                for image_file in direct_images:
                    image_path = os.path.join(base_path, image_file)
                    all_images.append({
                        'image_path': image_path,
                        'style': style_name
                    })
        else:
            # Multiple style folders
            available_folders = [f for f in os.listdir(base_path) 
                                 if os.path.isdir(os.path.join(base_path, f))]
            
            if style_folders is not None:
                # Process only specified folders
                folders_to_scan = [f for f in style_folders if f in available_folders]
                missing = [f for f in style_folders if f not in available_folders]
                
                if missing:
                    print(f"⚠ Warning: Folders not found: {missing}")
                
                if not folders_to_scan:
                    raise ValueError(f"None of the specified folders {style_folders} were found in {base_path}")
                
                print(f"Processing specified folders: {folders_to_scan}")
            else:
                # Process all folders
                folders_to_scan = available_folders
                print(f"Processing all available folders: {folders_to_scan}")
            
            for style_folder in folders_to_scan:
                style_path = os.path.join(base_path, style_folder)
                if os.path.isdir(style_path):
                    image_count = 0
                    for image_file in os.listdir(style_path):
                        if image_file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                            image_path = os.path.join(style_path, image_file)
                            all_images.append({
                                'image_path': image_path,
                                'style': style_folder
                            })
                            image_count += 1
                    print(f"  Found {image_count} images in '{style_folder}' folder")
        
        return pd.DataFrame(all_images)
    
    # Discover dataset with specified folders
    complete_dataset = discover_dataset(style_folders=style_folders_to_process)
    print(f"\nDiscovered dataset: {len(complete_dataset)} images")
    if len(complete_dataset) > 0:
        print(f"Style categories: {sorted(complete_dataset['style'].unique())}")
    else:
        print("Warning: No images found in dataset folder!")

# Display sample
if len(complete_dataset) > 0:
    print(f"\nFirst 5 images:")
    print(complete_dataset.head())
    print(f"\nTotal images to process: {len(complete_dataset)}")
    
    # Show breakdown by style
    print(f"\nImages by style:")
    style_counts = complete_dataset['style'].value_counts().sort_index()
    for style, count in style_counts.items():
        print(f"  {style}: {count} images")
else:
    print("ERROR: No images found! Please check your dataset folder.")


Complete dataset CSV not found. Discovering dataset from folder...
Processing specified folders: ['gal', 'girlish', 'kireime-casual']
  Found 954 images in 'gal' folder
  Found 1106 images in 'girlish' folder
  Found 1054 images in 'kireime-casual' folder

Discovered dataset: 3114 images
Style categories: ['gal', 'girlish', 'kireime-casual']

First 5 images:
                                          image_path style
0                          dataset/gal/108893606.jpg   gal
1                         dataset/gal/mig copy 4.jpg   gal
2  dataset/gal/smp_thum_da9b5a913036c6be9bbaa758e...   gal
3                   dataset/gal/code_197-450x730.jpg   gal
4  dataset/gal/da2058eb67200d32f426c45687e7ff9c_l...   gal

Total images to process: 3114

Images by style:
  gal: 954 images
  girlish: 1106 images
  kireime-casual: 1054 images


## 4. Caption Generation Functions


In [5]:
# Import conv_templates for caption generation
from llava.conversation import conv_templates

def generate_caption(image_path, style, model, tokenizer, image_processor, device):
    """
    Generate a detailed caption for a fashion image using LLaVA
    """
    try:
        # Load and preprocess image
        image = Image.open(image_path).convert('RGB')
        
        # Create conversation prompt for fashion description
        # Use conv_templates dictionary (imported from llava.conversation)
        conv_mode = "llava_v1"
        conv = conv_templates[conv_mode].copy()
        
        # Add image token to prompt
        from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
        prompt = "USER: " + DEFAULT_IMAGE_TOKEN + "\nDescribe this fashion image in detail, including clothing items, colors, style, and overall aesthetic. Focus on fashion elements that would help classify the style category.\nASSISTANT:"
        
        conv.append_message(conv.roles[0], prompt)
        conv.append_message(conv.roles[1], None)
        
        # Process image using LLaVA utilities
        from llava.mm_utils import process_images, tokenizer_image_token
        
        image_tensor = process_images([image], image_processor, model.config)
        image_tensor = image_tensor.to(device, dtype=torch.float16 if device == "cuda" else torch.float32)
        
        # Tokenize the prompt
        input_ids = tokenizer_image_token(
            prompt, 
            tokenizer, 
            IMAGE_TOKEN_INDEX, 
            return_tensors='pt'
        ).unsqueeze(0).to(device)
        
        # Generate response
        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                images=image_tensor,
                do_sample=True,
                temperature=0.7,
                max_new_tokens=256,
                use_cache=True
            )
        
        # Decode response - skip the input tokens
        output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        # Extract only the assistant's response
        if "ASSISTANT:" in output_text:
            response = output_text.split("ASSISTANT:")[-1].strip()
        else:
            response = output_text.strip()
        
        return {
            'image_path': image_path,
            'style': style,
            'caption': response,
            'status': 'success'
        }
        
    except Exception as e:
        import traceback
        error_msg = f"Error: {str(e)}"
        # For debugging, you can uncomment the line below
        # error_msg = f"Error: {str(e)}\n{traceback.format_exc()}"
        return {
            'image_path': image_path,
            'style': style,
            'caption': error_msg,
            'status': 'error'
        }

def batch_generate_captions(df, model, tokenizer, image_processor, device, batch_size=10, max_images=None):
    """
    Generate captions for a batch of images
    """
    if max_images:
        df = df.head(max_images)
    
    results = []
    total_batches = (len(df) + batch_size - 1) // batch_size
    
    print(f"Processing {len(df)} images in {total_batches} batches...")
    
    for i in tqdm(range(0, len(df), batch_size), desc="Generating captions"):
        batch_df = df.iloc[i:i+batch_size]
        batch_results = []
        
        for _, row in batch_df.iterrows():
            result = generate_caption(
                row['image_path'], 
                row['style'], 
                model, tokenizer, image_processor, device
            )
            batch_results.append(result)
        
        results.extend(batch_results)
        
        # Clear GPU cache periodically to avoid memory issues
        if torch.cuda.is_available() and (i // batch_size) % 5 == 0:
            torch.cuda.empty_cache()
        
        # Save intermediate results every batch
        if i % (batch_size * 5) == 0:  # Save every 5 batches
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(f'captions_temp_{i}.csv', index=False)
            print(f"Saved intermediate results: {len(results)} captions")
            
            # Display GPU memory usage
            if torch.cuda.is_available():
                print(f"GPU memory: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB allocated, "
                      f"{torch.cuda.memory_reserved(0) / 1024**3:.2f} GB reserved")
    
    # Final GPU cache clear
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return pd.DataFrame(results)


## 5. Test Caption Generation


In [6]:
# Test caption generation on a small sample first
print("Testing LLaVA caption generation on a small sample...")

# Check GPU memory before test
print_gpu_memory()

# Take a small sample for testing (2 images per style, or up to 5 total if single style)
if len(complete_dataset) > 0:
    if len(complete_dataset['style'].unique()) > 1:
        test_sample = complete_dataset.groupby('style').head(2).reset_index(drop=True)
    else:
        # Single style - just take first 3-5 images for testing
        test_sample = complete_dataset.head(5).reset_index(drop=True)
    
    print(f"Test sample: {len(test_sample)} images")
    print(f"Styles in test: {test_sample['style'].unique().tolist()}")
    
    # Generate captions for test sample
    test_results = batch_generate_captions(
        test_sample, 
        model, tokenizer, image_processor, device, 
        batch_size=2,  # Small batch for testing
        max_images=5
    )
    
    # Check GPU memory after test
    print("\nGPU memory after test:")
    print_gpu_memory()
    
    # Display results
    print("\n" + "="*70)
    print("Test Results:")
    print("="*70)
    for _, row in test_results.iterrows():
        print(f"\nStyle: {row['style']}")
        print(f"Image: {os.path.basename(row['image_path'])}")
        print(f"Status: {row['status']}")
        if row['status'] == 'success':
            print(f"Caption: {row['caption'][:300]}..." if len(row['caption']) > 300 else f"Caption: {row['caption']}")
        else:
            print(f"Error: {row['caption']}")
        print("-" * 70)
else:
    print("ERROR: No images to test! Please check your dataset.")


Testing LLaVA caption generation on a small sample...
GPU Memory - Allocated: 13.78 GB, Reserved: 13.79 GB
Test sample: 6 images
Styles in test: ['gal', 'girlish', 'kireime-casual']
Processing 5 images in 3 batches...


Generating captions:  33%|███▎      | 1/3 [00:04<00:09,  4.53s/it]

Saved intermediate results: 2 captions
GPU memory: 13.79 GB allocated, 13.81 GB reserved


Generating captions: 100%|██████████| 3/3 [00:11<00:00,  3.99s/it]


GPU memory after test:
GPU Memory - Allocated: 13.79 GB, Reserved: 13.81 GB

Test Results:

Style: gal
Image: 108893606.jpg
Status: success
Caption: The image features a beautiful young woman standing on a white background, wearing a black skirt and a white top with a black and white striped pattern. Her pose suggests confidence, and her lips give a slight pout. The black skirt adds a touch of elegance to her overall appearance, while the white ...
----------------------------------------------------------------------

Style: gal
Image: mig copy 4.jpg
Status: success
Caption: The image features a woman posing for the camera while wearing a white coat. She is also holding a black purse and a handbag, adding a touch of color to her outfit. Her overall style can be described as sophisticated and elegant.

The woman is standing next to a bench, giving an outdoor setting to t...
----------------------------------------------------------------------

Style: girlish
Image: 140912_book_15.jpg

## 6. Full Dataset Caption Generation


In [7]:
# Generate captions for the full dataset
if len(complete_dataset) == 0:
    print("ERROR: No images in dataset! Please check your dataset folder.")
else:
    print("Starting full dataset caption generation...")
    print(f"Total images to process: {len(complete_dataset)}")
    
    # Check GPU memory before starting
    print("\nGPU memory before processing:")
    print_gpu_memory()
    
    # Estimate processing time
    estimated_time_per_image = 2  # seconds (rough estimate)
    total_time_hours = (len(complete_dataset) * estimated_time_per_image) / 3600
    print(f"\nEstimated processing time: {total_time_hours:.1f} hours")
    
    # Process in smaller batches to avoid memory issues
    # Start with batch_size=5, reduce if you get CUDA OOM errors
    batch_size = 5  # Adjust based on your GPU memory
    print(f"Processing with batch size: {batch_size}")
    print("Note: Reduce batch_size if you encounter CUDA out of memory errors")
    
    # Generate captions
    all_captions = batch_generate_captions(
        complete_dataset,
        model, tokenizer, image_processor, device,
        batch_size=batch_size
    )
    
    # Check GPU memory after processing
    print("\nGPU memory after processing:")
    print_gpu_memory()
    
    print(f"\nCaption generation completed!")
    print(f"Total captions generated: {len(all_captions)}")
    print(f"Successful: {len(all_captions[all_captions['status'] == 'success'])}")
    print(f"Errors: {len(all_captions[all_captions['status'] == 'error'])}")


Starting full dataset caption generation...
Total images to process: 3114

GPU memory before processing:
GPU Memory - Allocated: 13.79 GB, Reserved: 13.81 GB

Estimated processing time: 1.7 hours
Processing with batch size: 5
Note: Reduce batch_size if you encounter CUDA out of memory errors
Processing 3114 images in 623 batches...


Generating captions:   0%|          | 1/623 [00:12<2:06:00, 12.15s/it]

Saved intermediate results: 5 captions
GPU memory: 13.79 GB allocated, 13.81 GB reserved


Generating captions:   1%|          | 6/623 [01:14<2:05:37, 12.22s/it]

Saved intermediate results: 30 captions
GPU memory: 13.79 GB allocated, 13.81 GB reserved


...

Saved intermediate results: 3080 captions
GPU memory: 13.79 GB allocated, 13.81 GB reserved


Generating captions: 100%|█████████▉| 621/623 [2:07:47<00:24, 12.30s/it]

Saved intermediate results: 3105 captions
GPU memory: 13.79 GB allocated, 13.81 GB reserved


Generating captions: 100%|██████████| 623/623 [2:08:07<00:00, 12.34s/it]


GPU memory after processing:
GPU Memory - Allocated: 13.79 GB, Reserved: 13.81 GB

Caption generation completed!
Total captions generated: 3114
Successful: 3114
Errors: 0


## 7. Save Results and Analysis


In [8]:
# Save captions to CSV
all_captions.to_csv('fashion_captions_llava.csv', index=False)
print("Captions saved to 'fashion_captions_llava.csv'")

# Save successful captions only
successful_captions = all_captions[all_captions['status'] == 'success']
successful_captions.to_csv('fashion_captions_llava_success.csv', index=False)
print(f"Successful captions saved to 'fashion_captions_llava_success.csv'")

# Save as JSON for easier integration with multi-modal framework
captions_json = []
for _, row in successful_captions.iterrows():
    captions_json.append({
        'image_path': row['image_path'],
        'style': row['style'],
        'caption': row['caption']
    })

with open('fashion_captions_llava.json', 'w') as f:
    json.dump(captions_json, f, indent=2)
print("Captions saved to 'fashion_captions_llava.json'")

# Analyze caption quality
print(f"\nCaption Analysis:")
print(f"Total images: {len(complete_dataset)}")
print(f"Successful captions: {len(successful_captions)}")
print(f"Success rate: {len(successful_captions)/len(complete_dataset)*100:.1f}%")

# Average caption length
avg_length = successful_captions['caption'].str.len().mean()
print(f"Average caption length: {avg_length:.0f} characters")

# Show sample captions by style
print(f"\nSample captions by style:")
for style in successful_captions['style'].unique()[:3]:
    style_captions = successful_captions[successful_captions['style'] == style]
    sample_caption = style_captions.iloc[0]['caption']
    print(f"\n{style.upper()}:")
    print(f"  {sample_caption[:200]}...")


Captions saved to 'fashion_captions_llava.csv'
Successful captions saved to 'fashion_captions_llava_success.csv'
Captions saved to 'fashion_captions_llava.json'

Caption Analysis:
Total images: 3114
Successful captions: 3114
Success rate: 100.0%
Average caption length: 512 characters

Sample captions by style:

GAL:
  In the image, a beautiful woman with long hair is standing against a white wall while wearing a striped shirt and a black skirt. The striped shirt features a combination of red, white, and black, crea...

GIRLISH:
  In the image, a young woman is wearing a white dress, which appears to be a strapless dress. She is also accessorizing with a scarf in shades of pink and purple, as well as carrying a brown purse. Her...

KIREIME-CASUAL:
  The image features a woman standing in front of a building, wearing a black jacket and a pink top. She is posing and seems to be the focal point of the photo. She has a handbag placed beside her, addi...


## 8. Memory Management and Optimization Tips

### GPU Memory Optimization:
- **Reduce batch size** if you get CUDA out of memory errors
- **Use 8-bit quantization**: Set `load_8bit=True` in model loading
- **Use 4-bit quantization**: Set `load_4bit=True` for even more memory savings
- **Process in smaller chunks**: Reduce `batch_size` parameter

### Performance Tips:
- **Monitor GPU usage**: Use `nvidia-smi` to check memory usage
- **Save intermediate results**: The notebook saves progress every 5 batches
- **Resume from checkpoint**: If interrupted, you can resume from saved temp files

### Expected Performance:
- **GPU (RTX 3080/4080)**: ~1-2 seconds per image
- **GPU (RTX 3060)**: ~2-3 seconds per image  
- **CPU only**: ~30-60 seconds per image (not recommended)

### Files Created:
- `fashion_captions_llava.csv`: All captions (including errors)
- `fashion_captions_llava_success.csv`: Only successful captions
- `fashion_captions_llava.json`: JSON format for easy integration
- `captions_temp_*.csv`: Intermediate saves (can be deleted after completion)

### Next Steps:
1. **Test with small sample first** (run test cell)
2. **Adjust batch size** based on your GPU memory
3. **Run full generation** (will take several hours)
4. **Use captions** in your multi-modal framework with CLIP + FashionBERT
